<a href="https://colab.research.google.com/github/rubayamist/DataScience/blob/main/soilclassification1.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Fixed from Claude**

The crash is almost certainly caused by loading the **entire dataset into RAM as a numpy array** (`SoilDataLoader.load_image_directory`) and then making several more full copies of it (normalization, train/val/test split, EDA). For soil image datasets (often thousands of images at 224×224×3), that easily balloons past Colab's ~12–13 GB system RAM — especially before you even get to the GPU.

The fix isn't a GPU-memory fix (T4 VRAM isn't the bottleneck here) — it's a **RAM** fix: stream images from disk in batches instead of materializing the whole dataset, and release memory between models.

Here's a rewritten pipeline that fixes this. I'll save it as a script you can paste into Colab cells.**Root cause:** it's a RAM crash, not a GPU/VRAM crash. `SoilDataLoader.load_image_directory()` loads *every* image into one giant numpy array, then normalization and the two `train_test_split()` calls each make additional full copies. Peak RAM ends up 3–5x the dataset size, which blows past Colab's ~12–13 GB system RAM ceiling — before training even starts.

**What the fixed pipeline above does differently:**

1. **Streams images from disk in batches** with `tf.keras.utils.image_dataset_from_directory` instead of loading the whole dataset into a numpy array — this is the fix that actually stops the crash.
2. **No EDA/preprocessing full-array copies** — normalization is done per-batch via each model's own `preprocess_input`.
3. **Class weights computed by counting files**, not loading images.
4. **Explicit cleanup between models**: `del model`, `tf.keras.backend.clear_session()`, `gc.collect()`, and closing matplotlib figures after each save — so training model 2 and 3 doesn't inherit model 1's memory footprint (this was likely a second contributor, since your loop kept `trained_models[model_name] = model` alive for every model).
5. **Mixed precision (`mixed_float16`)** — halves activation memory on the T4 and speeds up training.
6. Optional `max_images_per_class` cap to smoke-test the pipeline on a subset before a full run.
7. Optional snippet to copy the dataset from Drive to local Colab disk first — Drive I/O can be slow/flaky for many small file reads during training.

**If you still hit OOM after this**, in order of effort:
- Drop `batch_size` to 16.
- Cap `max_images_per_class` for a first pass.
- Use Colab's "High-RAM" runtime (Colab Pro, or free tier sometimes offers it) — Runtime → Change runtime type.
- Train one model per Colab session/runtime restart instead of looping through 3 in one session (safest option if RAM is very tight).
- Move to Kaggle Notebooks instead, which give ~30 GB RAM with a free GPU — often more forgiving for exactly this kind of pipeline.

The script assumes standard `ImageFolder`-style directory structure (one subfolder per class, `.jpg`/`.png` files) — same as your original. If your custom `SoilEDA`/`AdvancedPreprocessor` classes do something dataset-specific beyond normalization/class-weighting, let me know what and I can fold it into the streaming version without reintroducing the full in-memory array.

In [ ]:
# =====================================================================
# SOIL CLASSIFICATION PIPELINE — MEMORY-SAFE VERSION FOR COLAB (T4 GPU)
# =====================================================================
# WHY THE ORIGINAL CRASHED
# -------------------------------------------------------------------
# `SoilDataLoader.load_image_directory()` reads every image into ONE
# giant numpy array, all at once, in system RAM (not GPU VRAM).
# For N images at 224x224x3 float32 that's N * 224*224*3*4 bytes.
# 10,000 images alone = ~6 GB. Then the pipeline immediately makes
# several MORE full copies of that array:
#   - preprocessor.normalize()      -> new array (~2x)
#   - train_test_split() x2         -> new arrays (~another 1x-2x)
#   - EDA plotting on raw `images`  -> keeps the original alive too
# Peak RAM usage ends up at 3-5x the raw dataset size, which blows
# past Colab's free-tier ~12-13 GB system RAM ceiling. This is a RAM
# problem, not a GPU memory problem — switching GPU type won't fix it.
#
# THE FIX
# -------------------------------------------------------------------
# Never materialize the full dataset as one numpy array. Instead,
# stream images from disk in batches using tf.data / Keras utilities,
# and explicitly release memory (clear_session + gc.collect + del)
# between models so training model #2 doesn't inherit model #1's
# footprint.
# =====================================================================

import os
import gc
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, mixed_precision
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ---------------------------------------------------------------
# 0. GPU / MEMORY SETUP
# ---------------------------------------------------------------
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Mixed precision roughly halves activation memory on a T4 and speeds
# up training. Safe default for transfer-learning image models.
mixed_precision.set_global_policy('mixed_float16')

print(f"GPUs available: {len(gpus)}")
print(f"Mixed precision policy: {mixed_precision.global_policy()}")

# ---------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------
DATA_PATH = "/content/drive/MyDrive/DatasetRubaya/CyAUG-Dataset"

MODELS_TO_TRAIN = 'fast'   # 'all', 'fast', or a custom list

CONFIG = {
    'img_size': (224, 224),
    'batch_size': 32,          # lower to 16 if you still see OOM
    'epochs': 50,
    'patience': 8,
    'learning_rate': 1e-3,
    'validation_split': 0.2,   # taken out of the "train" portion below
    'test_split': 0.2,         # taken out of the full dataset
    'seed': 42,
    # If your dataset is huge and you just want to validate the
    # pipeline runs end-to-end before a full run, cap images per class:
    'max_images_per_class': None,   # e.g. 300 for a quick smoke test
}

data_path = Path(DATA_PATH)
if not data_path.exists():
    raise FileNotFoundError(f"Data path not found: {DATA_PATH}")

output_dir = Path('soil_classification_results')
output_dir.mkdir(exist_ok=True)

# ---------------------------------------------------------------
# 2. (OPTIONAL) BUILD A CAPPED WORKING COPY OF THE DATASET
# ---------------------------------------------------------------
# Loading straight from Google Drive with tf.data works but Drive I/O
# can be slow and flaky in Colab. If you hit "Input/Output error" or
# very slow first epochs, copy the dataset to local Colab disk first:
#
#   !rm -rf /content/soil_data
#   !cp -r "{DATA_PATH}" /content/soil_data
#   data_path = Path("/content/soil_data")
#
# This does NOT load images into RAM — it's a disk-to-disk copy, so it
# won't cause an OOM, only takes a few minutes depending on dataset size.

if CONFIG['max_images_per_class']:
    import shutil
    capped_path = Path('/content/soil_data_capped')
    if capped_path.exists():
        shutil.rmtree(capped_path)
    capped_path.mkdir(parents=True)
    for class_dir in data_path.iterdir():
        if not class_dir.is_dir():
            continue
        dest = capped_path / class_dir.name
        dest.mkdir()
        files = sorted(list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')))
        for f in files[:CONFIG['max_images_per_class']]:
            shutil.copy(f, dest / f.name)
    data_path = capped_path
    print(f"✓ Using capped dataset at {data_path} "
          f"({CONFIG['max_images_per_class']} images/class max)")

# ---------------------------------------------------------------
# 3. STREAMED DATASETS (no full-array load into RAM)
# ---------------------------------------------------------------
# image_dataset_from_directory reads images off disk batch-by-batch
# during training, not all at once. This is the single biggest fix.

full_train_ds = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=CONFIG['test_split'],
    subset='training',
    seed=CONFIG['seed'],
    image_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=CONFIG['test_split'],
    subset='validation',
    seed=CONFIG['seed'],
    image_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
)

class_names = full_train_ds.class_names
num_classes = len(class_names)
print(f"\n✓ Found {num_classes} soil types: {class_names}")

# Split the "training" portion further into train/val
train_batches = tf.data.experimental.cardinality(full_train_ds).numpy()
val_batches = int(train_batches * CONFIG['validation_split'])
val_ds = full_train_ds.take(val_batches)
train_ds = full_train_ds.skip(val_batches)

print(f"✓ Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"✓ Val batches:   {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"✓ Test batches:  {tf.data.experimental.cardinality(test_ds).numpy()}")

# ---------------------------------------------------------------
# 4. CLASS WEIGHTS (computed by counting files, not loading images)
# ---------------------------------------------------------------
class_counts = {}
for cname in class_names:
    n = len(list((data_path / cname).glob('*.jpg'))) + \
        len(list((data_path / cname).glob('*.png')))
    class_counts[cname] = n

total = sum(class_counts.values())
class_weight = {
    i: total / (num_classes * class_counts[cname])
    for i, cname in enumerate(class_names)
}
print("\n✓ Class weights (for imbalance):")
for i, cname in enumerate(class_names):
    print(f"   {cname}: {class_weight[i]:.3f}  ({class_counts[cname]} images)")

# ---------------------------------------------------------------
# 5. PERFORMANCE PIPELINE (cache/prefetch, augmentation, normalization)
# ---------------------------------------------------------------
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# NOTE: no manual normalization step here — each model's own
# `preprocess_input` (imagenet-style) is applied inside the model
# builder below, so we never keep a separately-normalized full copy
# of the dataset in memory.

def prep(ds, training, preprocess_fn):
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)
    # Deliberately NOT using .cache() here: caching the decoded/augmented
    # dataset in RAM reintroduces the same OOM risk for large datasets.
    # If your dataset is small (<2-3k images) you can safely add
    # `.cache()` right before `.prefetch(AUTOTUNE)` for a speed boost.

# ---------------------------------------------------------------
# 6. MODEL FACTORY (transfer learning, memory-conscious)
# ---------------------------------------------------------------
MODEL_BUILDERS = {
    'mobilenetv2': (tf.keras.applications.MobileNetV2,
                     tf.keras.applications.mobilenet_v2.preprocess_input),
    'efficientnetb0': (tf.keras.applications.EfficientNetB0,
                         tf.keras.applications.efficientnet.preprocess_input),
    'resnet50': (tf.keras.applications.ResNet50,
                  tf.keras.applications.resnet50.preprocess_input),
    'densenet121': (tf.keras.applications.DenseNet121,
                      tf.keras.applications.densenet.preprocess_input),
}

def build_model(name, input_shape, num_classes):
    base_cls, preprocess_fn = MODEL_BUILDERS[name]
    base = base_cls(include_top=False, weights='imagenet', input_shape=input_shape)
    base.trainable = False  # start frozen; unfreeze later for fine-tuning if desired

    inputs = tf.keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = tf.keras.Model(inputs, outputs)
    return model, preprocess_fn

if MODELS_TO_TRAIN == 'all':
    model_list = ['mobilenetv2', 'efficientnetb0', 'resnet50', 'densenet121']
elif MODELS_TO_TRAIN == 'fast':
    model_list = ['mobilenetv2', 'efficientnetb0', 'resnet50']
else:
    model_list = MODELS_TO_TRAIN

# ---------------------------------------------------------------
# 7. TRAIN LOOP — with explicit cleanup between models
# ---------------------------------------------------------------
results_summary = {}

for idx, model_name in enumerate(model_list, 1):
    print(f"\n{'='*80}\n[{idx}/{len(model_list)}] TRAINING: {model_name.upper()}\n{'='*80}")

    try:
        model, preprocess_fn = build_model(
            model_name, CONFIG['img_size'] + (3,), num_classes)
        print(f"✓ Model created: {model.count_params():,} params")

        train_ready = prep(train_ds, training=True, preprocess_fn=preprocess_fn)
        val_ready = prep(val_ds, training=False, preprocess_fn=preprocess_fn)
        test_ready = prep(test_ds, training=False, preprocess_fn=preprocess_fn)

        model.compile(
            optimizer=tf.keras.optimizers.Adam(CONFIG['learning_rate']),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'],
        )

        ckpt_path = output_dir / f'{model_name}_best.keras'
        cb = [
            callbacks.EarlyStopping(monitor='val_loss',
                                     patience=CONFIG['patience'],
                                     restore_best_weights=True),
            callbacks.ModelCheckpoint(str(ckpt_path), monitor='val_loss',
                                       save_best_only=True),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
        ]

        history = model.fit(
            train_ready,
            validation_data=val_ready,
            epochs=CONFIG['epochs'],
            class_weight=class_weight,
            callbacks=cb,
            verbose=1,
        )

        test_loss, test_acc = model.evaluate(test_ready, verbose=0)
        print(f"✓ {model_name} test accuracy: {test_acc:.4f}")

        # Confusion matrix / classification report on the (small) test set
        y_true, y_pred = [], []
        for xb, yb in test_ready:
            preds = model.predict(xb, verbose=0)
            y_true.extend(yb.numpy())
            y_pred.extend(np.argmax(preds, axis=1))

        report = classification_report(y_true, y_pred, target_names=class_names,
                                        output_dict=True, zero_division=0)
        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix — {model_name}')
        plt.ylabel('True'); plt.xlabel('Predicted')
        plt.tight_layout()
        plt.savefig(output_dir / f'{model_name}_confusion_matrix.png')
        plt.close()  # IMPORTANT: close figures, don't let them pile up in RAM

        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(history.history['accuracy'], label='train')
        plt.plot(history.history['val_accuracy'], label='val')
        plt.title('Accuracy'); plt.legend()
        plt.subplot(1, 2, 2)
        plt.plot(history.history['loss'], label='train')
        plt.plot(history.history['val_loss'], label='val')
        plt.title('Loss'); plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / f'{model_name}_history.png')
        plt.close()

        results_summary[model_name] = {
            'test_accuracy': float(test_acc),
            'test_loss': float(test_loss),
            'macro_f1': report['macro avg']['f1-score'],
        }

        # Save just the metrics + best weights to disk — do NOT keep the
        # full model object alive in a `trained_models[name] = model`
        # dict, that's what silently accumulates RAM across the loop.
        with open(output_dir / f'{model_name}_report.json', 'w') as f:
            json.dump(report, f, indent=2)

        print(f"✓ {model_name} complete. Best weights saved to {ckpt_path}")

    except Exception as e:
        print(f"\n❌ Error training {model_name}: {e}")
        import traceback
        traceback.print_exc()

    finally:
        # --- THE KEY MEMORY CLEANUP STEP ---
        # Without this, TF/Keras layers, optimizer state, and the
        # augmentation/prefetch graphs from model N stay resident while
        # model N+1 starts building, and RAM climbs every iteration
        # until Colab kills the session.
        del model, train_ready, val_ready, test_ready
        tf.keras.backend.clear_session()
        gc.collect()

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
for name, res in results_summary.items():
    print(f"  {name:20s} test_acc={res['test_accuracy']:.4f} "
          f"macro_f1={res['macro_f1']:.4f}")

GPUs available: 1
Mixed precision policy: <DTypePolicy "mixed_float16">
Found 5103 files belonging to 7 classes.
Using 4083 files for training.
Found 5103 files belonging to 7 classes.
Using 1020 files for validation.

✓ Found 7 soil types: ['Alluvial_Soil', 'Arid_Soil', 'Black_Soil', 'Laterite_Soil', 'Mountain_Soil', 'Red_Soil', 'Yellow_Soil']
✓ Train batches: 103
✓ Val batches:   25
✓ Test batches:  32

✓ Class weights (for imbalance):
   Alluvial_Soil: 3.400  (50 images)
   Arid_Soil: 0.599  (284 images)
   Black_Soil: 0.656  (259 images)
   Laterite_Soil: 0.776  (219 images)
   Mountain_Soil: 0.846  (201 images)
   Red_Soil: 1.574  (108 images)
   Yellow_Soil: 2.464  (69 images)

[1/3] TRAINING: MOBILENETV2
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✓ Model created: 2,266,951 params
Epoch 1/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 144s 945ms/step - accuracy: 0.6442 - loss: 1.5084 - val_accuracy: 0.8550 - val_loss: 0.4688 - learning_rate: 0.0010
Epoch 2/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 

Good catch — the risk in these cells isn't RAM from images anymore (that's fixed), it's a different memory trap: **`trained_models[model_name] = model`** keeps every trained Keras model (with its full frozen backbone) resident simultaneously for the ensemble step, and `X_val`/`X_test` are still full in-memory numpy arrays in the original code. Since we switched to `tf.data` streaming, those numpy arrays no longer exist, and I'll avoid ever holding multiple models in memory at once by having each model save its **prediction probabilities to disk** right after training — the ensemble step then just averages small `.npy` arrays instead of reloading models.

Let me extend the pipeline file to do this end-to-end.**What changed in Steps 4–8, and why it stops the crash:**

1. **No more `trained_models[name] = model` dict.** That was the second big memory leak — it kept every trained model (backbone + head) alive simultaneously just so Step 5 could ensemble them. Now each model, right after training, predicts once on val/test and saves the probability arrays (`{model}_val_probs.npy`, `{model}_test_probs.npy` — tiny, just `n_samples × n_classes`) to disk, then is deleted (`del model` + `clear_session()` + `gc.collect()`, same as Step 4 before).

2. **Ensemble (Step 5) is now just `np.mean()` over saved `.npy` files** — no models are loaded back into memory at all. Soft-voting on 3 tiny arrays instead of 3 loaded neural nets.

3. **Final test evaluation (Step 6)** picks the best model by validation accuracy and reuses its already-saved test probabilities — no re-prediction, no reloading.

4. **Model comparison (Step 7)** builds the `pandas` DataFrame straight from the lightweight `results_summary` dict (numbers only, not model objects).

5. **Saving (Step 8)** copies the best model's `.keras` checkpoint file (already written by `ModelCheckpoint` during training) instead of holding a model object in RAM to call `.save()` on it. I switched `.h5` → `.keras`, the current recommended Keras format — let me know if you specifically need `.h5` for compatibility elsewhere.

One thing worth double-checking on your end: this relies on `val_ds`/`test_ds` iterating in the **same order** every time they're consumed (across different models), since predictions and true labels are collected separately and matched positionally. `image_dataset_from_directory` shuffles the file list once at construction (fixed by `seed`), not per-epoch, so this holds — but if you add any `.shuffle()` call yourself to `val_ready`/`test_ready` anywhere, that assumption breaks silently (accuracy would look wrong, not crash). Don't add shuffling to those two pipelines.

In [ ]:
# =====================================================================
# SOIL CLASSIFICATION PIPELINE — MEMORY-SAFE VERSION FOR COLAB (T4 GPU)
# =====================================================================
# WHY THE ORIGINAL CRASHED
# -------------------------------------------------------------------
# `SoilDataLoader.load_image_directory()` reads every image into ONE
# giant numpy array, all at once, in system RAM (not GPU VRAM).
# For N images at 224x224x3 float32 that's N * 224*224*3*4 bytes.
# 10,000 images alone = ~6 GB. Then the pipeline immediately makes
# several MORE full copies of that array:
#   - preprocessor.normalize()      -> new array (~2x)
#   - train_test_split() x2         -> new arrays (~another 1x-2x)
#   - EDA plotting on raw `images`  -> keeps the original alive too
# Peak RAM usage ends up at 3-5x the raw dataset size, which blows
# past Colab's free-tier ~12-13 GB system RAM ceiling. This is a RAM
# problem, not a GPU memory problem — switching GPU type won't fix it.
#
# THE FIX
# -------------------------------------------------------------------
# Never materialize the full dataset as one numpy array. Instead,
# stream images from disk in batches using tf.data / Keras utilities,
# and explicitly release memory (clear_session + gc.collect + del)
# between models so training model #2 doesn't inherit model #1's
# footprint.
# =====================================================================

import os
import gc
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, mixed_precision
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ---------------------------------------------------------------
# 0. GPU / MEMORY SETUP
# ---------------------------------------------------------------
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Mixed precision roughly halves activation memory on a T4 and speeds
# up training. Safe default for transfer-learning image models.
mixed_precision.set_global_policy('mixed_float16')

print(f"GPUs available: {len(gpus)}")
print(f"Mixed precision policy: {mixed_precision.global_policy()}")

# ---------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------
DATA_PATH = "/content/drive/MyDrive/DatasetRubaya/CyAUG-Dataset"

MODELS_TO_TRAIN = 'fast'   # 'all', 'fast', or a custom list

CONFIG = {
    'img_size': (224, 224),
    'batch_size': 32,          # lower to 16 if you still see OOM
    'epochs': 50,
    'patience': 8,
    'learning_rate': 1e-3,
    'validation_split': 0.2,   # taken out of the "train" portion below
    'test_split': 0.2,         # taken out of the full dataset
    'seed': 42,
    # If your dataset is huge and you just want to validate the
    # pipeline runs end-to-end before a full run, cap images per class:
    'max_images_per_class': None,   # e.g. 300 for a quick smoke test
}

data_path = Path(DATA_PATH)
if not data_path.exists():
    raise FileNotFoundError(f"Data path not found: {DATA_PATH}")

output_dir = Path('soil_classification_results')
output_dir.mkdir(exist_ok=True)

# ---------------------------------------------------------------
# 2. (OPTIONAL) BUILD A CAPPED WORKING COPY OF THE DATASET
# ---------------------------------------------------------------
# Loading straight from Google Drive with tf.data works but Drive I/O
# can be slow and flaky in Colab. If you hit "Input/Output error" or
# very slow first epochs, copy the dataset to local Colab disk first:
#
#   !rm -rf /content/soil_data
#   !cp -r "{DATA_PATH}" /content/soil_data
#   data_path = Path("/content/soil_data")
#
# This does NOT load images into RAM — it's a disk-to-disk copy, so it
# won't cause an OOM, only takes a few minutes depending on dataset size.

if CONFIG['max_images_per_class']:
    import shutil
    capped_path = Path('/content/soil_data_capped')
    if capped_path.exists():
        shutil.rmtree(capped_path)
    capped_path.mkdir(parents=True)
    for class_dir in data_path.iterdir():
        if not class_dir.is_dir():
            continue
        dest = capped_path / class_dir.name
        dest.mkdir()
        files = sorted(list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')))
        for f in files[:CONFIG['max_images_per_class']]:
            shutil.copy(f, dest / f.name)
    data_path = capped_path
    print(f"✓ Using capped dataset at {data_path} "
          f"({CONFIG['max_images_per_class']} images/class max)")

# ---------------------------------------------------------------
# 3. STREAMED DATASETS (no full-array load into RAM)
# ---------------------------------------------------------------
# image_dataset_from_directory reads images off disk batch-by-batch
# during training, not all at once. This is the single biggest fix.

full_train_ds = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=CONFIG['test_split'],
    subset='training',
    seed=CONFIG['seed'],
    image_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=CONFIG['test_split'],
    subset='validation',
    seed=CONFIG['seed'],
    image_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
)

class_names = full_train_ds.class_names
num_classes = len(class_names)
print(f"\n✓ Found {num_classes} soil types: {class_names}")

# Split the "training" portion further into train/val
train_batches = tf.data.experimental.cardinality(full_train_ds).numpy()
val_batches = int(train_batches * CONFIG['validation_split'])
val_ds = full_train_ds.take(val_batches)
train_ds = full_train_ds.skip(val_batches)

print(f"✓ Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"✓ Val batches:   {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"✓ Test batches:  {tf.data.experimental.cardinality(test_ds).numpy()}")

# ---------------------------------------------------------------
# 4. CLASS WEIGHTS (computed by counting files, not loading images)
# ---------------------------------------------------------------
class_counts = {}
for cname in class_names:
    n = len(list((data_path / cname).glob('*.jpg'))) + \
        len(list((data_path / cname).glob('*.png')))
    class_counts[cname] = n

total = sum(class_counts.values())
class_weight = {
    i: total / (num_classes * class_counts[cname])
    for i, cname in enumerate(class_names)
}
print("\n✓ Class weights (for imbalance):")
for i, cname in enumerate(class_names):
    print(f"   {cname}: {class_weight[i]:.3f}  ({class_counts[cname]} images)")

# ---------------------------------------------------------------
# 5. PERFORMANCE PIPELINE (cache/prefetch, augmentation, normalization)
# ---------------------------------------------------------------
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# NOTE: no manual normalization step here — each model's own
# `preprocess_input` (imagenet-style) is applied inside the model
# builder below, so we never keep a separately-normalized full copy
# of the dataset in memory.

def prep(ds, training, preprocess_fn):
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)
    # Deliberately NOT using .cache() here: caching the decoded/augmented
    # dataset in RAM reintroduces the same OOM risk for large datasets.
    # If your dataset is small (<2-3k images) you can safely add
    # `.cache()` right before `.prefetch(AUTOTUNE)` for a speed boost.

# ---------------------------------------------------------------
# 6. MODEL FACTORY (transfer learning, memory-conscious)
# ---------------------------------------------------------------
MODEL_BUILDERS = {
    'mobilenetv2': (tf.keras.applications.MobileNetV2,
                     tf.keras.applications.mobilenet_v2.preprocess_input),
    'efficientnetb0': (tf.keras.applications.EfficientNetB0,
                         tf.keras.applications.efficientnet.preprocess_input),
    'resnet50': (tf.keras.applications.ResNet50,
                  tf.keras.applications.resnet50.preprocess_input),
    'densenet121': (tf.keras.applications.DenseNet121,
                      tf.keras.applications.densenet.preprocess_input),
}

def build_model(name, input_shape, num_classes):
    base_cls, preprocess_fn = MODEL_BUILDERS[name]
    base = base_cls(include_top=False, weights='imagenet', input_shape=input_shape)
    base.trainable = False  # start frozen; unfreeze later for fine-tuning if desired

    inputs = tf.keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = tf.keras.Model(inputs, outputs)
    return model, preprocess_fn

if MODELS_TO_TRAIN == 'all':
    model_list = ['mobilenetv2', 'efficientnetb0', 'resnet50', 'densenet121']
elif MODELS_TO_TRAIN == 'fast':
    model_list = ['mobilenetv2', 'efficientnetb0', 'resnet50']
else:
    model_list = MODELS_TO_TRAIN

# ---------------------------------------------------------------
# 7. TRAIN LOOP — with explicit cleanup between models
# ---------------------------------------------------------------
results_summary = {}

for idx, model_name in enumerate(model_list, 1):
    print(f"\n{'='*80}\n[{idx}/{len(model_list)}] TRAINING: {model_name.upper()}\n{'='*80}")

    try:
        model, preprocess_fn = build_model(
            model_name, CONFIG['img_size'] + (3,), num_classes)
        print(f"✓ Model created: {model.count_params():,} params")

        train_ready = prep(train_ds, training=True, preprocess_fn=preprocess_fn)
        val_ready = prep(val_ds, training=False, preprocess_fn=preprocess_fn)
        test_ready = prep(test_ds, training=False, preprocess_fn=preprocess_fn)

        model.compile(
            optimizer=tf.keras.optimizers.Adam(CONFIG['learning_rate']),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'],
        )

        ckpt_path = output_dir / f'{model_name}_best.keras'
        cb = [
            callbacks.EarlyStopping(monitor='val_loss',
                                     patience=CONFIG['patience'],
                                     restore_best_weights=True),
            callbacks.ModelCheckpoint(str(ckpt_path), monitor='val_loss',
                                       save_best_only=True),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
        ]

        history = model.fit(
            train_ready,
            validation_data=val_ready,
            epochs=CONFIG['epochs'],
            class_weight=class_weight,
            callbacks=cb,
            verbose=1,
        )

        test_loss, test_acc = model.evaluate(test_ready, verbose=0)
        print(f"✓ {model_name} test accuracy: {test_acc:.4f}")

        # --- Save prediction probabilities to disk instead of keeping the
        # model object around for ensembling later. These arrays are tiny
        # (n_samples x n_classes), so this can never cause an OOM no matter
        # how many models you train. ---
        def _collect_probs(ds):
            probs_list, true_list = [], []
            for xb, yb in ds:
                probs_list.append(model.predict(xb, verbose=0))
                true_list.append(yb.numpy())
            return np.concatenate(probs_list), np.concatenate(true_list)

        val_probs, val_true = _collect_probs(val_ready)
        test_probs, test_true = _collect_probs(test_ready)

        np.save(output_dir / f'{model_name}_val_probs.npy', val_probs)
        np.save(output_dir / f'{model_name}_test_probs.npy', test_probs)
        # True labels are identical across models (same fixed dataset order),
        # so only write them once.
        if not (output_dir / 'y_val_true.npy').exists():
            np.save(output_dir / 'y_val_true.npy', val_true)
        if not (output_dir / 'y_test_true.npy').exists():
            np.save(output_dir / 'y_test_true.npy', test_true)

        y_true, y_pred = test_true, np.argmax(test_probs, axis=1)
        report = classification_report(y_true, y_pred, target_names=class_names,
                                        output_dict=True, zero_division=0)
        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix — {model_name}')
        plt.ylabel('True'); plt.xlabel('Predicted')
        plt.tight_layout()
        plt.savefig(output_dir / f'{model_name}_confusion_matrix.png')
        plt.close()  # IMPORTANT: close figures, don't let them pile up in RAM

        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(history.history['accuracy'], label='train')
        plt.plot(history.history['val_accuracy'], label='val')
        plt.title('Accuracy'); plt.legend()
        plt.subplot(1, 2, 2)
        plt.plot(history.history['loss'], label='train')
        plt.plot(history.history['val_loss'], label='val')
        plt.title('Loss'); plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / f'{model_name}_history.png')
        plt.close()

        results_summary[model_name] = {
            'val_accuracy': float((np.argmax(val_probs, axis=1) == val_true).mean()),
            'test_accuracy': float(test_acc),
            'test_loss': float(test_loss),
            'macro_f1': report['macro avg']['f1-score'],
            'checkpoint': str(ckpt_path),
        }

        # Save just the metrics + best weights to disk — do NOT keep the
        # full model object alive in a `trained_models[name] = model`
        # dict, that's what silently accumulates RAM across the loop.
        with open(output_dir / f'{model_name}_report.json', 'w') as f:
            json.dump(report, f, indent=2)

        print(f"✓ {model_name} complete. Best weights saved to {ckpt_path}")

    except Exception as e:
        print(f"\n❌ Error training {model_name}: {e}")
        import traceback
        traceback.print_exc()

    finally:
        # --- THE KEY MEMORY CLEANUP STEP ---
        # Without this, TF/Keras layers, optimizer state, and the
        # augmentation/prefetch graphs from model N stay resident while
        # model N+1 starts building, and RAM climbs every iteration
        # until Colab kills the session.
        for _name in ('model', 'train_ready', 'val_ready', 'test_ready'):
            if _name in dir():
                exec(f'del {_name}')
        tf.keras.backend.clear_session()
        gc.collect()

print("\n" + "="*80)
print("STEP 4 SUMMARY")
print("="*80)
for name, res in results_summary.items():
    print(f"  {name:20s} val_acc={res['val_accuracy']:.4f} "
          f"test_acc={res['test_accuracy']:.4f} macro_f1={res['macro_f1']:.4f}")


# =====================================================================
# STEP 5: ENSEMBLE — built purely from saved probability arrays.
# No model is ever reloaded here, so this step cannot cause an OOM
# regardless of how many models were trained above.
# =====================================================================
print("\n" + "="*80)
print("STEP 5: ENSEMBLE METHODS")
print("="*80)

y_val_true = np.load(output_dir / 'y_val_true.npy')
y_test_true = np.load(output_dir / 'y_test_true.npy')

available_models = [m for m in model_list if (output_dir / f'{m}_val_probs.npy').exists()]

if len(available_models) >= 2:
    val_probs_list = [np.load(output_dir / f'{m}_val_probs.npy') for m in available_models]
    test_probs_list = [np.load(output_dir / f'{m}_test_probs.npy') for m in available_models]

    ensemble_val_probs = np.mean(val_probs_list, axis=0)   # soft voting
    ensemble_test_probs = np.mean(test_probs_list, axis=0)

    y_val_pred_ens = np.argmax(ensemble_val_probs, axis=1)
    y_test_pred_ens = np.argmax(ensemble_test_probs, axis=1)

    ens_val_acc = float((y_val_pred_ens == y_val_true).mean())
    ens_test_acc = float((y_test_pred_ens == y_test_true).mean())
    ens_report = classification_report(y_test_true, y_test_pred_ens,
                                        target_names=class_names,
                                        output_dict=True, zero_division=0)

    print(f"✓ Ensemble ({' + '.join(available_models)})")
    print(f"   Val accuracy:  {ens_val_acc:.4f}")
    print(f"   Test accuracy: {ens_test_acc:.4f}")

    cm_ens = confusion_matrix(y_test_true, y_test_pred_ens)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix — Ensemble (soft voting)')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(output_dir / 'ensemble_confusion_matrix.png', dpi=200, bbox_inches='tight')
    plt.close()

    results_summary['ensemble'] = {
        'val_accuracy': ens_val_acc,
        'test_accuracy': ens_test_acc,
        'macro_f1': ens_report['macro avg']['f1-score'],
        'models_used': available_models,
    }
    print("✓ Ensemble evaluation complete!")
else:
    print("⚠ Fewer than 2 successfully trained models — skipping ensemble.")


# =====================================================================
# STEP 6: FINAL TEST SET EVALUATION
# =====================================================================
print("\n" + "="*80)
print("STEP 6: FINAL TEST SET EVALUATION")
print("="*80)

candidates = {k: v for k, v in results_summary.items() if 'val_accuracy' in v}

if candidates:
    best_name = max(candidates.items(), key=lambda kv: kv[1]['val_accuracy'])[0]
    best_info = candidates[best_name]
    print(f"\n🏆 Best model (by validation accuracy): {best_name}")
    print(f"📊 Test accuracy: {best_info['test_accuracy']:.4f}")

    if best_name == 'ensemble':
        y_test_pred_best = y_test_pred_ens
    else:
        best_test_probs = np.load(output_dir / f'{best_name}_test_probs.npy')
        y_test_pred_best = np.argmax(best_test_probs, axis=1)

    print(f"\n{classification_report(y_test_true, y_test_pred_best, target_names=class_names, zero_division=0)}")

    cm_test = confusion_matrix(y_test_true, y_test_pred_best)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Test Confusion Matrix — {best_name}')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(output_dir / 'test_confusion_matrix_best.png', dpi=300, bbox_inches='tight')
    plt.close()
else:
    print("\n⚠ No models were successfully trained.")
    best_name = "None"
    best_info = {'test_accuracy': 0.0}


# =====================================================================
# STEP 7: MODEL COMPARISON
# =====================================================================
print("\n" + "="*80)
print("STEP 7: MODEL COMPARISON")
print("="*80)

import pandas as pd

comparison_df = pd.DataFrame(results_summary).T
keep_cols = [c for c in ['val_accuracy', 'test_accuracy', 'macro_f1'] if c in comparison_df.columns]
comparison_df = comparison_df[keep_cols].sort_values('test_accuracy', ascending=False)
print(comparison_df)
comparison_df.to_csv(output_dir / 'model_comparison.csv')
print(f"\n✓ Saved: {output_dir / 'model_comparison.csv'}")


# =====================================================================
# STEP 8: SAVING RESULTS
# =====================================================================
print("\n" + "="*80)
print("STEP 8: SAVING RESULTS")
print("="*80)

import shutil
from datetime import datetime

if candidates and best_name != 'ensemble':
    src_ckpt = output_dir / f'{best_name}_best.keras'
    if src_ckpt.exists():
        dst = output_dir / f'final_best_model_{best_name}.keras'
        shutil.copy(src_ckpt, dst)
        print(f"✓ Saved: {dst.name}")
elif best_name == 'ensemble':
    print("✓ Best result is the ensemble — its component checkpoints are already "
          f"saved individually ({', '.join(available_models)}); combine their "
          "predictions (soft-vote average) at inference time.")

final_summary = {
    'best_model': best_name,
    'test_accuracy': float(best_info['test_accuracy']) if candidates else 0.0,
    'configuration': CONFIG,
    'class_names': class_names,
    'num_classes': num_classes,
    'all_results': results_summary,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}
with open(output_dir / 'results_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=4)
print("✓ Saved: results_summary.json")

print("\n" + "="*80)
print("✅ PIPELINE COMPLETE!")
print("="*80)
if candidates:
    print(f"\n🏆 Best Model: {best_name}")
    print(f"📊 Test Accuracy: {best_info['test_accuracy']:.4f}")
    print(f"\n📁 All results saved in: {output_dir.absolute()}")
else:
    print("\n⚠ No models were successfully trained. Check errors above.")

In [14]:
# ========== CONFIGURATION ==========
#DATA_PATH = "/kaggle/input/comprehensive-soil-classification-datasets/CyAUG-Dataset"
DATA_PATH = "/content/drive/MyDrive/DatasetRubaya/CyAUG-Dataset"

# Choose: 'all', 'fast', or custom list
MODELS_TO_TRAIN = 'fast'  # Fast models for quick testing

CONFIG = {
    'img_size': (224, 224),
    'batch_size': 32,
    'epochs': 50,
    'patience': 10,  # Early stopping patience
    'learning_rate': 0.001,
    'validation_split': 0.2,
    'test_split': 0.2,
    'optimizer': 'adam',
    'normalization': 'imagenet',
}

print("\n" + "="*80)
print("COMPREHENSIVE SOIL CLASSIFICATION PIPELINE - FIXED VERSION")
print("="*80)
print(f"\n📋 Configuration:")
print(f"   Data Path: {DATA_PATH}")
print(f"   Models: {MODELS_TO_TRAIN}")
print(f"   Image Size: {CONFIG['img_size']}")
print(f"   Batch Size: {CONFIG['batch_size']}")
print(f"   Epochs: {CONFIG['epochs']}")
print(f"   Patience: {CONFIG['patience']}")


COMPREHENSIVE SOIL CLASSIFICATION PIPELINE - FIXED VERSION

📋 Configuration:
   Data Path: /content/drive/MyDrive/DatasetRubaya/CyAUG-Dataset
   Models: fast
   Image Size: (224, 224)
   Batch Size: 32
   Epochs: 50
   Patience: 10


In [15]:
# ========== VERIFY DATA PATH ==========
data_path = Path(DATA_PATH)

if not data_path.exists():
    print(f"\n❌ ERROR: Data path not found: {DATA_PATH}")
    print("Please update DATA_PATH variable")
    exit(1)

soil_types = [d.name for d in data_path.iterdir() if d.is_dir()]
print(f"\n✓ Found {len(soil_types)} soil types:")
for soil_type in sorted(soil_types)[:10]:  # Show first 10
    num_images = len(list((data_path / soil_type).glob('*.jpg'))) + \
                    len(list((data_path / soil_type).glob('*.png')))
    print(f"   - {soil_type}: {num_images} images")


✓ Found 7 soil types:
   - Alluvial_Soil: 50 images
   - Arid_Soil: 284 images
   - Black_Soil: 259 images
   - Laterite_Soil: 219 images
   - Mountain_Soil: 201 images
   - Red_Soil: 108 images
   - Yellow_Soil: 69 images


In [16]:
# ========== CREATE OUTPUT DIRECTORY ==========
output_dir = Path('soil_classification_results')
output_dir.mkdir(exist_ok=True)
os.chdir(output_dir)
print(f"\n✓ Output directory: {output_dir.absolute()}")


✓ Output directory: /content/soil_classification_results/soil_classification_results


In [17]:
# Step 1: Load Data
print("\n" + "="*80)
print("STEP 1: DATA LOADING")
print("="*80)

loader = SoilDataLoader(data_path)
images, labels, class_names = loader.load_image_directory(img_size=CONFIG['img_size'])


STEP 1: DATA LOADING
✓ Found 7 classes: ['Alluvial_Soil', 'Arid_Soil', 'Black_Soil', 'Laterite_Soil', 'Mountain_Soil', 'Red_Soil', 'Yellow_Soil']
   Loading Alluvial_Soil: 693 images...
   Loading Arid_Soil: 284 images...
   Loading Black_Soil: 1177 images...
   Loading Laterite_Soil: 219 images...
   Loading Mountain_Soil: 201 images...
   Loading Red_Soil: 1128 images...
   Loading Yellow_Soil: 1401 images...
✓ Successfully loaded 5103 images


In [18]:
# Step 2: EDA
print("\n" + "="*80)
print("STEP 2: EXPLORATORY DATA ANALYSIS")
print("="*80)

eda = SoilEDA(images, labels, class_names)
eda.analyze_all()


STEP 2: EXPLORATORY DATA ANALYSIS

EXPLORATORY DATA ANALYSIS

📊 BASIC STATISTICS
--------------------------------------------------
Total samples: 5103
Number of classes: 7
Image shape: (224, 224, 3)
Data type: uint8
Value range: [0, 255]

📈 CLASS DISTRIBUTION
--------------------------------------------------
Alluvial_Soil: 693 (13.58%)
Arid_Soil: 284 (5.57%)
Black_Soil: 1177 (23.06%)
Laterite_Soil: 219 (4.29%)
Mountain_Soil: 201 (3.94%)
Red_Soil: 1128 (22.10%)
Yellow_Soil: 1401 (27.45%)

Imbalance Ratio: 6.97

🖼️  IMAGE PROPERTIES
--------------------------------------------------
Brightness - Mean: 104.16, Std: 41.29
Contrast - Mean: 54.16, Std: 19.18

🎨 COLOR ANALYSIS
--------------------------------------------------
Red - Mean: 148.97, Std: 70.35
Green - Mean: 100.91, Std: 59.91
Blue - Mean: 62.59, Std: 52.29

🔍 TEXTURE ANALYSIS
--------------------------------------------------
Edge Intensity - Mean: 73.9567, Std: 24.9631

EDA SUMMARY
✓ EDA complete. Visualizations saved.


In [ ]:
# Step 3: Preprocessing
print("\n" + "="*80)
print("STEP 3: DATA PREPROCESSING")
print("="*80)

preprocessor = AdvancedPreprocessor(images, labels)
images = preprocessor.normalize(method=CONFIG['normalization'])
class_weights = preprocessor.handle_imbalance()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    images, labels,
    test_size=CONFIG['test_split'],
    stratify=labels,
    random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=CONFIG['validation_split'],
    stratify=y_train,
    random_state=42
)

print(f"\n✓ Dataset Split:")
print(f"   Train: {len(X_train)} ({len(X_train)/len(images)*100:.1f}%)")
print(f"   Validation: {len(X_val)} ({len(X_val)/len(images)*100:.1f}%)")
print(f"   Test: {len(X_test)} ({len(X_test)/len(images)*100:.1f}%)")


STEP 3: DATA PREPROCESSING


In [ ]:
# Step 4: Model Training
print("\n" + "="*80)
print("STEP 4: MODEL TRAINING")
print("="*80)

if MODELS_TO_TRAIN == 'all':
    model_list = ['mobilenetv2', 'efficientnetb0', 'resnet50', 'densenet121', 'inceptionv3', 'xception', 'custom_cnn']
elif MODELS_TO_TRAIN == 'fast':
    model_list = ['mobilenetv2', 'efficientnetb0', 'resnet50']
else:
    model_list = MODELS_TO_TRAIN

print(f"\n📋 Models to train: {', '.join(model_list)}")

factory = ModelFactory(input_shape=images.shape[1:], num_classes=len(class_names))
trained_models = {}
evaluator = ModelEvaluator(class_names)

for idx, model_name in enumerate(model_list, 1):
    try:
        print(f"\n{'='*80}")
        print(f"[{idx}/{len(model_list)}] TRAINING: {model_name.upper()}")
        print(f"{'='*80}")

        model = factory.create_model(model_name, pretrained=True)
        print(f"✓ Model created: {model.count_params():,} parameters")

        trainer = ModelTrainer(model, model_name, X_train, y_train, X_val, y_val, class_weights)
        trainer.compile_model(optimizer=CONFIG['optimizer'], learning_rate=CONFIG['learning_rate'])
        trainer.train(epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'], patience=CONFIG['patience'])

        results = trainer.evaluate()
        evaluator.evaluate_model(model_name, y_val, results['predictions'],
                                results['probabilities'], results['history'])

        evaluator.print_results(model_name)
        evaluator.plot_confusion_matrix(model_name)
        evaluator.plot_training_history(model_name)

        trained_models[model_name] = model
        print(f"\n✓ {model_name} training complete!")

    except Exception as e:
        print(f"\n❌ Error training {model_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue


STEP 4: MODEL TRAINING

📋 Models to train: mobilenetv2, efficientnetb0, resnet50

[1/3] TRAINING: MOBILENETV2


I0000 00:00:1760268673.994263      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✓ Model created: 3,054,151 parameters

Training mobilenetv2
Epoch 1/50


I0000 00:00:1760268731.091696      62 service.cc:148] XLA service 0x78209c0036d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1760268731.092564      62 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1760268735.540924      62 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1760268742.917865      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760268743.114765      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


  1/102 ━━━━━━━━━━━━━━━━━━━━ 2:08:51 77s/step - accuracy: 0.0938 - loss: 2.2886 - top3_accuracy: 0.4375

...

102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step - accuracy: 0.5775 - loss: 1.3179 - top3_accuracy: 0.8024
Epoch 1: val_accuracy improved from -inf to 0.40931, saving model to best_mobilenetv2.h5
102/102 ━━━━━━━━━━━━━━━━━━━━ 125s 477ms/step - accuracy: 0.5789 - loss: 1.3150 - top3_accuracy: 0.8034 - val_accuracy: 0.4093 - val_loss: 5.7818 - val_top3_accuracy: 0.8015 - learning_rate: 0.0010
Epoch 2/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.8384 - loss: 0.6397 - top3_accuracy: 0.9562
Epoch 2: val_accuracy improved from 0.40931 to 0.61765, saving model to best_mobilenetv2.h5
102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 96ms/step - accuracy: 0.8384 - loss: 0.6395 - top3_accuracy: 0.9562 - val_accuracy: 0.6176 - val_loss: 8.7473 - val_top3_accuracy: 0.7255 - learning_rate: 0.0010
Epoch 3/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.8785 - loss: 0.4546 - top3_accuracy: 0.9703
Epoch 3: val_accuracy did not improve from 0.61765
102/102 ━━━━━━━━━━━━━━━━━━━━ 9s 91ms/step - accuracy:

E0000 00:00:1760269069.704521      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760269069.894071      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760269070.390197      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760269070.596302      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760269070.971085      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:0

101/102 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.6324 - loss: 1.1823 - top3_accuracy: 0.8228

E0000 00:00:1760269123.575316      63 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760269123.765331      63 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760269124.246386      63 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760269124.455364      63 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1760269124.828607      63 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:0

102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 525ms/step - accuracy: 0.6337 - loss: 1.1790 - top3_accuracy: 0.8238
Epoch 1: val_accuracy improved from -inf to 0.23897, saving model to best_efficientnetb0.h5
102/102 ━━━━━━━━━━━━━━━━━━━━ 185s 714ms/step - accuracy: 0.6350 - loss: 1.1758 - top3_accuracy: 0.8248 - val_accuracy: 0.2390 - val_loss: 2.4285 - val_top3_accuracy: 0.4326 - learning_rate: 0.0010
Epoch 2/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.8661 - loss: 0.5333 - top3_accuracy: 0.9683
Epoch 2: val_accuracy improved from 0.23897 to 0.29167, saving model to best_efficientnetb0.h5
102/102 ━━━━━━━━━━━━━━━━━━━━ 14s 135ms/step - accuracy: 0.8662 - loss: 0.5328 - top3_accuracy: 0.9683 - val_accuracy: 0.2917 - val_loss: 2.1198 - val_top3_accuracy: 0.5196 - learning_rate: 0.0010
Epoch 3/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.9203 - loss: 0.2914 - top3_accuracy: 0.9845
Epoch 3: val_accuracy did not improve from 0.29167
102/102 ━━━━━━━━━━━━━━━━━━━━ 13s 126ms/step 

In [ ]:
# Step 5: Ensemble (if multiple models)
if len(trained_models) >= 2:
    print("\n" + "="*80)
    print("STEP 5: ENSEMBLE METHODS")
    print("="*80)

    ensemble = EnsembleModel(list(trained_models.values()), list(trained_models.keys()))
    y_pred_ensemble, y_pred_probs_ensemble = ensemble.predict_voting(X_val, method='soft')
    evaluator.evaluate_model('Ensemble_Voting', y_val, y_pred_ensemble, y_pred_probs_ensemble)
    evaluator.print_results('Ensemble_Voting')
    evaluator.plot_confusion_matrix('Ensemble_Voting')
    print("✓ Ensemble evaluation complete!")


STEP 5: ENSEMBLE METHODS

EVALUATION RESULTS: Ensemble_Voting
Accuracy: 0.9779
Precision: 0.9795
Recall: 0.9779
F1-Score: 0.9778
Cohen's Kappa: 0.9724
Matthews Correlation: 0.9725
ROC AUC: 0.9993

               precision    recall  f1-score   support

Alluvial_Soil       0.97      0.98      0.98       111
    Arid_Soil       0.93      0.89      0.91        46
   Black_Soil       1.00      0.99      1.00       188
Laterite_Soil       0.96      0.77      0.86        35
Mountain_Soil       0.80      1.00      0.89        32
     Red_Soil       0.99      1.00      0.99       180
  Yellow_Soil       1.00      0.99      0.99       224

     accuracy                           0.98       816
    macro avg       0.95      0.95      0.95       816
 weighted avg       0.98      0.98      0.98       816

✓ Ensemble evaluation complete!


In [ ]:
# Step 6: Test Set Evaluation
print("\n" + "="*80)
print("STEP 6: FINAL TEST SET EVALUATION")
print("="*80)

if trained_models:
    best_model_name = max(evaluator.results.items(), key=lambda x: x[1]['accuracy'])[0]
    best_model = trained_models.get(best_model_name)

    if best_model:
        print(f"\n🏆 Best Model: {best_model_name}")

        y_test_pred_probs = best_model.predict(X_test, verbose=0)
        y_test_pred = np.argmax(y_test_pred_probs, axis=1)

        test_accuracy = accuracy_score(y_test, y_test_pred)
        print(f"📊 Test Accuracy: {test_accuracy:.4f}")
        print(f"\n{classification_report(y_test, y_test_pred, target_names=class_names, zero_division=0)}")

        # Test confusion matrix
        cm_test = confusion_matrix(y_test, y_test_pred)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Test Confusion Matrix - {best_model_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig('test_confusion_matrix.png', dpi=300, bbox_inches='tight')
        plt.close()
else:
    print("\n⚠ No models were successfully trained")
    best_model = None
    best_model_name = "None"
    test_accuracy = 0.0


STEP 6: FINAL TEST SET EVALUATION


In [ ]:
# Step 7: Comparison
print("\n" + "="*80)
print("STEP 7: MODEL COMPARISON")
print("="*80)

comparison_df = evaluator.compare_models()


STEP 7: MODEL COMPARISON

MODEL COMPARISON
          Model  Accuracy  Precision   Recall  F1-Score  Cohen Kappa  Matthews Corr
Ensemble_Voting  0.977941   0.979483 0.977941  0.977779     0.972353       0.972467
 efficientnetb0  0.974265   0.974307 0.974265  0.973544     0.967698       0.967794
       resnet50  0.960784   0.962216 0.960784  0.961353     0.950951       0.951001
    mobilenetv2  0.764706   0.851243 0.764706  0.773751     0.710675       0.729998


In [ ]:
# Step 8: Save Results
print("\n" + "="*80)
print("STEP 8: SAVING RESULTS")
print("="*80)

if trained_models and best_model:
    best_model.save(f'final_best_model_{best_model_name}.h5')
    print(f"✓ Saved: final_best_model_{best_model_name}.h5")

results_summary = {
    'best_model': best_model_name if trained_models else 'None',
    'test_accuracy': float(test_accuracy) if trained_models and best_model else 0.0,
    'configuration': CONFIG,
    'class_names': class_names,
    'num_classes': len(class_names),
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open('results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=4)
print("✓ Saved: results_summary.json")


STEP 8: SAVING RESULTS
✓ Saved: results_summary.json


In [ ]:
# Final Summary
print("\n" + "="*80)
print("✅ PIPELINE COMPLETE!")
print("="*80)
if trained_models and best_model:
    print(f"\n🏆 Best Model: {best_model_name}")
    print(f"📊 Test Accuracy: {test_accuracy:.4f}")
    print(f"\n📁 All results saved in: {output_dir.absolute()}")
else:
    print("\n⚠ No models were successfully trained. Check errors above.")
    print(f"📁 Available outputs in: {output_dir.absolute()}")


✅ PIPELINE COMPLETE!

⚠ No models were successfully trained. Check errors above.
📁 Available outputs in: /kaggle/working/soil_classification_results/soil_classification_results
